# Fraud Payment Detection - 종합 분류 모델 비교 (모범 답안)

이 파일은 결제 사기 탐지 문제를 여러 분류 모델로 해결하고 성능을 비교하는 종합 문제입니다.
수강생들이 자신의 문제 풀이와 비교하여 학습할 수 있도록 작성되었습니다.

**학습 목표:**
- 여러 분류 모델(Logistic Regression, KNN, Decision Tree, Random Forest)을 동일한 데이터에 적용
- 각 모델의 성능을 비교하고 분석
- 사기 탐지 문제에 가장 적합한 모델 선택

**데이터 설명:**
- accountAgeDays : 계정이 생성된 기간 (일)
- numItems : 구매한 항목 수
- localTime : 결제가 이루어진 시기 (부동 숫자로 변환 됨)
- paymentMethod : 결제 방법 (페이팔, 상점 신용 카드 또는 신용 카드)
- paymentMethodAgeDays : 결제가 완료된 기간 (일)
- label 0 - 정상, 1 - fraud

## 1. 데이터 로드 및 전처리

필요한 라이브러리를 import 합니다. 여러 분류 모델과 평가 지표를 사용합니다.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, f1_score, accuracy_score, 
                          precision_score, recall_score, roc_auc_score)
import seaborn as sns

결제 사기 탐지 데이터셋을 로드합니다. 이 데이터로 정상 결제와 사기 결제를 구분하는 모델을 학습합니다.

In [3]:
df = pd.read_csv("datasets/payment_fraud.csv")
df.sample(5)

,accountAgeDays,numItems,localTime,paymentMethod,paymentMethodAgeDays,label
3532,1214,1,4.886641,creditcard,0.001389,0
24790,476,1,4.895263,creditcard,0.000000,0
21301,2000,1,4.748314,creditcard,0.018750,0
33752,480,1,5.017904,paypal,479.011806,0
33559,20,1,4.921349,creditcard,0.000000,0


라벨의 분포를 확인합니다. 정상(0)과 사기(1)의 비율을 확인할 수 있습니다. 데이터 불균형이 있을 수 있습니다.

In [4]:
df.label.value_counts()

label
0    38661
1      560
Name: count, dtype: int64

데이터 불균형 문제를 해결하기 위해 사기(1) 샘플을 모두 선택하고, 정상(0) 샘플은 일부만 샘플링합니다.

In [5]:
df1 = df[df.label == 1]
df1.shape

(560, 6)

정상 결제 데이터에서 1000개를 무작위로 샘플링합니다. 이렇게 하면 클래스 불균형을 완화할 수 있습니다.

In [6]:
df0 = df[df.label == 0].sample(1000, random_state=0)
df0.shape

(1000, 6)

샘플링된 정상 데이터와 전체 사기 데이터를 합쳐서 새로운 데이터셋을 만듭니다.

In [7]:
df_new = pd.concat([df0, df1])
df_new.shape

(1560, 6)

In [8]:
df_new.label.value_counts()

label
0    1000
1     560
Name: count, dtype: int64

결제 방법을 원-핫 인코딩으로 변환합니다. 각 결제 방법이 별도의 이진 특성으로 변환됩니다.

In [ ]:
# 'paymentMethod' 컬럼을 원-핫 인코딩(one-hot encoding)으로 변환
# 각 결제 수단 값이 새로운 더미 변수(0/1 컬럼)로 생성됨


원-핫 인코딩이 적용된 데이터를 확인합니다. paymentMethod 컬럼이 여러 개의 이진 컬럼으로 분리되었습니다.

In [ ]:
# 데이터프레임(df_new)에서 임의로 5개 행을 무작위 추출하여 확인
# 데이터 구조나 전처리 결과를 빠르게 점검할 때 사용


특성과 라벨을 분리합니다. `pop()` 메서드를 사용하여 label 컬럼을 제거하면서 동시에 y로 할당합니다.

In [ ]:
# 'label' 컬럼을 데이터프레임에서 분리하여 타깃 변수(y)로 저장
# pop()을 사용하면 df_new에서는 해당 컬럼이 제거됨


# 나머지 컬럼들을 입력 특성(feature)로 사용하기 위해
# DataFrame을 NumPy 배열 형태로 변환


데이터를 훈련용과 테스트용으로 분할합니다. test_size=0.2로 80%는 훈련, 20%는 테스트 데이터로 나뉩니다.

In [ ]:
# 전체 데이터(X, y)를 학습용(train)과 테스트용(test)으로 분할
# test_size=0.2  → 전체 데이터의 20%를 테스트 데이터로 사용
# random_state=0 → 매번 동일한 분할 결과를 얻기 위해 난수 시드 고정


# 분할된 데이터의 차원(shape)을 확인
# X_train, X_test : (샘플 수, 특성 수)
# y_train, y_test : (샘플 수,)



특성 스케일링을 수행합니다. StandardScaler를 사용하여 모든 특성을 평균 0, 표준편차 1로 정규화합니다.
일부 모델(KNN, Logistic Regression)은 스케일링이 중요합니다.

In [ ]:
# StandardScaler 객체 생성
# 각 특성을 평균 0, 표준편차 1이 되도록 표준화(z-score 정규화) 수행


# 학습 데이터(X_train)에 대해 평균과 표준편차를 학습한 뒤 변환
# fit_transform() : 학습(fit) + 변환(transform)을 한 번에 수행


# 테스트 데이터(X_test)는 학습 데이터에서 계산된
# 평균과 표준편차를 그대로 사용하여 변환만 수행
# (데이터 누수(data leakage) 방지를 위함)



## 2. 여러 분류 모델 적용 및 평가

### 2.1 Logistic Regression

로지스틱 회귀 모델을 생성하고 훈련 데이터로 학습시킵니다. 선형 분류 모델로 빠르고 해석이 용이합니다.

In [ ]:
# 로지스틱 회귀(Logistic Regression) 분류기 생성
# random_state=0 : 모델 학습 시 난수 요소를 고정하여 재현성 확보


# 표준화된 학습 데이터(X_train_scaled)와 정답 레이블(y_train)을 사용해
# 로지스틱 회귀 분류 모델 학습



테스트 데이터에 대한 예측을 수행합니다.

In [ ]:
# 학습된 로지스틱 회귀 모델을 사용하여
# 테스트 데이터(X_test_scaled)에 대한 예측값 생성


# 테스트 세트에 실제로 존재하는 사기 거래(라벨=1)의 개수 출력


# 모델이 사기 거래로 예측한 거래(예측값=1)의 개수 출력


# 테스트 세트에서의 분류 정확도(Accuracy) 계산 및 출력
# 정확도 = (정확히 맞춘 예측 수) / (전체 테스트 데이터 수)



### 2.2 K-Nearest Neighbors (KNN)

KNN 분류기를 생성하고 학습시킵니다. KNN은 거리 기반 분류 모델로, 스케일링이 중요합니다.

In [ ]:
# K-최근접 이웃(K-Nearest Neighbors, KNN) 분류기 생성
# n_neighbors=5 : 분류 시 가장 가까운 5개의 이웃을 기준으로 클래스 결정


# 표준화된 학습 데이터(X_train_scaled)와 정답 레이블(y_train)을 사용해
# KNN 분류 모델 학습(저장)



테스트 데이터에 대한 예측을 수행합니다.

In [ ]:
# 학습된 KNN 분류 모델을 사용하여
# 테스트 데이터(X_test_scaled)에 대한 예측값 생성


# 테스트 세트에 실제로 존재하는 사기 거래(라벨=1)의 개수 출력


# KNN 모델이 사기 거래로 예측한 거래(예측값=1)의 개수 출력


# 테스트 세트에서의 분류 정확도(Accuracy) 계산 및 출력
# 정확도 = (정확히 맞춘 예측 수) / (전체 테스트 데이터 수)



### 2.3 Decision Tree

Decision Tree 분류기를 생성하고 학습시킵니다. 트리 모델은 스케일링이 필요 없으며, 해석이 용이합니다.

In [ ]:
# 의사결정나무(Decision Tree) 분류기 생성
# max_depth=5  : 트리의 최대 깊이를 5로 제한하여 과적합(overfitting) 방지
# random_state=0 : 학습 과정의 난수를 고정하여 재현성 확보


# 학습 데이터(X_train)와 정답 레이블(y_train)을 사용해
# 의사결정나무 분류 모델 학습
# ※ 결정트리는 특성 스케일링이 필수가 아니므로 원본 데이터 사용



테스트 데이터에 대한 예측을 수행합니다.

In [ ]:
# 학습된 의사결정나무 모델을 사용하여
# 테스트 데이터(X_test)에 대한 예측값 생성
# ※ 결정트리는 스케일링이 필요 없으므로 원본 테스트 데이터 사용


# 테스트 세트에 실제로 존재하는 사기 거래(라벨=1)의 개수 출력


# 의사결정나무 모델이 사기 거래로 예측한 거래(예측값=1)의 개수 출력


# 테스트 세트에서의 분류 정확도(Accuracy) 계산 및 출력
# 정확도 = (정확히 맞춘 예측 수) / (전체 테스트 데이터 수)



### 2.4 Random Forest

Random Forest 분류기를 생성하고 학습시킵니다. 여러 의사결정 트리를 앙상블하여 더 강건한 모델을 만듭니다.

In [ ]:
# 랜덤 포레스트(Random Forest) 분류기 생성
# n_estimators=200 : 생성할 결정트리의 개수 (클수록 성능 안정, 학습 시간 증가)
# max_depth=None   : 각 트리를 끝까지 성장시켜 복잡한 패턴까지 학습
# random_state=0   : 난수 시드를 고정하여 결과 재현성 확보


# 학습 데이터(X_train)와 정답 레이블(y_train)을 사용해
# 랜덤 포레스트 분류 모델 학습
# ※ 트리 기반 모델이므로 특성 스케일링 없이 원본 데이터 사용



테스트 데이터에 대한 예측을 수행합니다.

In [ ]:
# 학습된 랜덤 포레스트 모델을 사용하여
# 테스트 데이터(X_test)에 대한 예측값 생성
# ※ 트리 기반 모델이므로 스케일링되지 않은 원본 데이터 사용


# 테스트 세트에 실제로 존재하는 사기 거래(라벨=1)의 개수 출력


# 랜덤 포레스트 모델이 사기 거래로 예측한 거래(예측값=1)의 개수 출력


# 테스트 세트에서의 분류 정확도(Accuracy) 계산 및 출력
# 정확도 = (정확히 맞춘 예측 수) / (전체 테스트 데이터 수)



## 3. 모델 성능 비교

각 모델의 성능 지표를 계산하여 비교합니다. 사기 탐지에서는 Recall(재현율)이 특히 중요합니다.

In [ ]:
# 각 모델의 예측 결과를 하나의 딕셔너리로 정리
# key   : 모델 이름
# value : 해당 모델의 테스트 데이터 예측값
models = {
    'Logistic Regression': y_pred_lr,
    'KNN': y_pred_knn,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf
}

# 모델별 성능 지표를 저장할 리스트
results = []

# 각 모델에 대해 성능 지표 계산
for model_name, y_pred in models.items():
    # 정확도(Accuracy): 전체 예측 중 맞게 예측한 비율
    acc = accuracy_score(y_test, y_pred)
    
    # 정밀도(Precision): 사기로 예측한 것 중 실제 사기인 비율
    # zero_division=0 : 분모가 0이 되는 경우 오류 대신 0 반환
    prec = precision_score(y_test, y_pred, zero_division=0)
    
    # 재현율(Recall): 실제 사기 거래 중 모델이 맞게 찾아낸 비율
    rec = recall_score(y_test, y_pred, zero_division=0)
    
    # F1 Score: 정밀도와 재현율의 조화 평균 (불균형 데이터에서 중요)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # 계산된 성능 지표를 딕셔너리 형태로 저장
    results.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1
    })

# 리스트를 DataFrame으로 변환
results_df = pd.DataFrame(results)

# F1 Score 기준으로 내림차순 정렬
results_df = results_df.sort_values('F1 Score', ascending=False)

# 결과 출력
print("\n=== 모델 성능 비교 ===\n")
print(results_df.to_string(index=False))